# Store 1 | Customer Data Cleaning

**Preparing customer records for an e-commerce loyalty program, and a first look at who the customers are.**

*Practice project developed during a Data Science training program. Author: Abner Álvarez Cabrera.*

---

### Resumen en español

**Contexto:** Store 1, una tienda en línea, prepara un programa de fidelización y necesita datos de clientes confiables.

**Qué se hizo:** se escribió una función de limpieza que estandariza nombres, edades y categorías y se aplicó igual a todos los registros. Después, se respondieron preguntas básicas de negocio: ingreso total, clientes menores de 30 años, clientes jóvenes con gasto alto y compradores por categoría.

**Hallazgos principales (muestra de 10 clientes):**
- Ingreso total registrado: 9,189.
- 5 de 10 clientes tienen menos de 30 años, pero solo 2 de ellos gastaron más de 1,000.
- Ropa (`clothes`) y hogar (`home`) aparecen cada una en 5 de los 10 clientes.

**Aprendizaje clave:** un proceso de limpieza estandarizado en una sola función evita errores que aparecen cuando el mismo código se copia y se ajusta a mano para cada registro.

**Limitación:** los datos son una muestra pequeña de práctica; los resultados describen esta muestra y no permiten conclusiones generales.

---

## 1. Business context

Store 1 is an online store preparing to launch a **customer loyalty program**. To design personalized campaigns, the marketing team needs customer records that are consistent and reliable.

The raw records have formatting problems that would break grouping, filtering and counting. This notebook:

1. Builds a **single, reusable cleaning function** and applies it to every record.
2. Uses the clean data to answer the team's first questions:
   - What is the total revenue recorded?
   - Which customers are under 30?
   - Which young customers are also high spenders?
   - Who buys in a given category?

## 2. The data

Each customer is stored as a nested list with five fields:

| Index | Field | Example | Problem found |
|---|---|---|---|
| 0 | `user_id` | `'32415'` | — |
| 1 | `user_name` | `' mike_reed '` | Extra spaces, underscores, inconsistent capitalization |
| 2 | `user_age` | `32.0` | Stored as a decimal instead of an integer |
| 3 | `fav_categories` | `['ELECTRONICS', 'SPORT']` | All uppercase |
| 4 | `total_spendings` | `[894, 213]` | Amount per category, in the same order as the categories |

The dataset is a small sample of 10 customers provided for the exercise.

In [1]:
users_raw = [
    ['32415', ' mike_reed ', 32.0, ['ELECTRONICS', 'SPORT', 'BOOKS'], [894, 213, 173]],
    ['31980', 'kate morgan', 24.0, ['CLOTHES', 'BOOKS'], [439, 390]],
    ['32156', ' john doe ', 37.0, ['ELECTRONICS', 'HOME', 'FOOD'], [459, 120, 99]],
    ['32761', 'SAMANTHA SMITH', 29.0, ['CLOTHES', 'ELECTRONICS', 'BEAUTY'], [299, 679, 85]],
    ['32984', 'David White', 41.0, ['BOOKS', 'HOME', 'SPORT'], [234, 329, 243]],
    ['33001', 'emily brown', 26.0, ['BEAUTY', 'HOME', 'FOOD'], [213, 659, 79]],
    ['33767', ' Maria Garcia', 33.0, ['CLOTHES', 'FOOD', 'BEAUTY'], [499, 189, 63]],
    ['33912', 'JOSE MARTINEZ', 22.0, ['SPORT', 'ELECTRONICS', 'HOME'], [259, 549, 109]],
    ['34009', 'lisa wilson ', 35.0, ['HOME', 'BOOKS', 'CLOTHES'], [329, 189, 329]],
    ['34278', 'James Lee', 28.0, ['BEAUTY', 'CLOTHES', 'ELECTRONICS'], [189, 299, 579]]
]

print(f"Customers in the dataset: {len(users_raw)}")

Customers in the dataset: 10


## 3. Data cleaning

### 3.1 A single cleaning function

All the cleaning rules live in one function, so every record goes through exactly the same steps:

- **Name:** remove outer spaces, convert to lowercase, replace underscores with spaces, and split into `[first_name, last_name]`.
- **Age:** convert from decimal to integer.
- **Categories:** convert to lowercase so they can be compared and filtered reliably.

The function returns a new list and leaves the raw record unchanged.

In [2]:
def clean_user(user):
    # Standardize the name and split it into [first_name, last_name]
    name = user[1].strip().lower().replace("_", " ").split()

    # Convert the age from decimal to integer
    age = int(user[2])

    # Convert every category to lowercase
    categories = []
    for cat in user[3]:
        categories.append(cat.lower())

    # Return a new record with the same structure as the original
    return [user[0], name, age, categories, user[4]]

### 3.2 Test on one record

In [3]:
test_user = ['32415', ' mike_reed ', 32.0, ['ELECTRONICS', 'SPORT', 'BOOKS'], [894, 213, 173]]

print("Before:", test_user)
print("After: ", clean_user(test_user))

Before: ['32415', ' mike_reed ', 32.0, ['ELECTRONICS', 'SPORT', 'BOOKS'], [894, 213, 173]]
After:  ['32415', ['mike', 'reed'], 32, ['electronics', 'sport', 'books'], [894, 213, 173]]


### 3.3 Apply the function to every customer

In [4]:
users_clean = []

for user in users_raw:
    users_clean.append(clean_user(user))

for user in users_clean:
    print(user)

['32415', ['mike', 'reed'], 32, ['electronics', 'sport', 'books'], [894, 213, 173]]
['31980', ['kate', 'morgan'], 24, ['clothes', 'books'], [439, 390]]
['32156', ['john', 'doe'], 37, ['electronics', 'home', 'food'], [459, 120, 99]]
['32761', ['samantha', 'smith'], 29, ['clothes', 'electronics', 'beauty'], [299, 679, 85]]
['32984', ['david', 'white'], 41, ['books', 'home', 'sport'], [234, 329, 243]]
['33001', ['emily', 'brown'], 26, ['beauty', 'home', 'food'], [213, 659, 79]]
['33767', ['maria', 'garcia'], 33, ['clothes', 'food', 'beauty'], [499, 189, 63]]
['33912', ['jose', 'martinez'], 22, ['sport', 'electronics', 'home'], [259, 549, 109]]
['34009', ['lisa', 'wilson'], 35, ['home', 'books', 'clothes'], [329, 189, 329]]
['34278', ['james', 'lee'], 28, ['beauty', 'clothes', 'electronics'], [189, 299, 579]]


All 10 records now share the same format: names split into lowercase first and last name, ages as integers and categories in lowercase.

## 4. Customer analysis

### 4.1 Total revenue

In [5]:
revenue = 0

for user in users_clean:
    # The last field holds the amount spent per category
    revenue += sum(user[4])

print(f"Total revenue recorded: {revenue}")

Total revenue recorded: 9189


### 4.2 Customers under 30

The marketing team wants to target young customers with specific campaigns.

In [6]:
for user in users_clean:
    if user[2] < 30:
        print(user[1][0], user[2])

kate 24
samantha 29
emily 26
jose 22
james 28


### 4.3 Young, high-spending customers

Customers under 30 **and** with a total spend above 1,000 are candidates for special benefits.

In [7]:
for user in users_clean:
    total_spent = sum(user[4])
    if user[2] < 30 and total_spent > 1000:
        print(user[1][0], total_spent)

samantha 1063
james 1067


### 4.4 Customers who buy clothes

In [8]:
for user in users_clean:
    if "clothes" in user[3]:
        print(user[1][0], user[2])

kate 24
samantha 29
maria 33
lisa 35
james 28


### 4.5 A reusable filter by category

Instead of rewriting the loop for every category, this function receives the category as a parameter and returns each matching customer's ID, full name, age and total spend.

In [9]:
def get_clients_by_category(users, category):
    result = []
    for user in users:
        if category in user[3]:
            # ID, full name, age and total spend
            result.append([user[0], user[1], user[2], sum(user[4])])
    return result


home_clients = get_clients_by_category(users_clean, "home")

for client in home_clients:
    print(client)

['32156', ['john', 'doe'], 37, 678]
['32984', ['david', 'white'], 41, 806]
['33001', ['emily', 'brown'], 26, 951]
['33912', ['jose', 'martinez'], 22, 917]
['34009', ['lisa', 'wilson'], 35, 847]


## 5. Key findings

*Based on a sample of 10 customers.*

- **Total revenue recorded: 9,189.**
- **Half of the customers (5 of 10) are under 30**, but only **2 of them** (Samantha and James) spent more than 1,000. Age alone is not enough to identify the most valuable young customers; spending needs to be part of the segmentation.
- **Clothes and home each appear for 5 of the 10 customers**, which makes them natural starting points for category campaigns.
- The category filter is reusable: the same function can list buyers for any category without new code.

**Limitation:** the sample is small and was created for practice, so these results describe this dataset only and should not be read as general business conclusions.

## 6. Key learnings

- **Standardize the process, not each case.** In an earlier version of this exercise, the cleaning code was copied and adjusted by hand for each customer, and one copy missed a step. The output still looked correct only because those names had no underscores, so the error was silent. Moving the rules into `clean_user()` guarantees that every record is processed the same way. This is the same principle as standardizing a production process in industrial engineering.
- **Reusable functions scale.** `get_clients_by_category()` answers a new business question by changing one argument.
- **Clean before you analyze.** Lowercasing names and categories is what makes filters such as `"clothes" in user[3]` reliable.

## 7. Tools

Python (lists, loops, conditionals, functions, f-strings) · Jupyter Notebook